### Research Agent 



1. Loads `engineering_model_iso15926.json` and extracts items
then 
2. Focuses on **3 domains innicially **: criticality safety · radiation protection · PUREX process
then 
3. For each item runs a real LangChain ReAct agent with **Claude API LLM call**

then 
4. Search tools use **ArXiv API** (papers) + **DuckDuckGo HTML** (standards/vendors)
then

5. Displays full results: standards found · technologies · gap analysis

### Requirements
- `ANTHROPIC_API_KEY` in your `.env` (only key needed)
- Internet access (ArXiv + DuckDuckGo)
- Python packages: `langchain langchain-anthropic pydantic httpx python-dotenv`

```

---
## CELL 1 — Install & imports

In [1]:
import os, json, re, time, textwrap, urllib.request, urllib.parse, ssl, xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Dict, List, Optional
from collections import Counter
from uuid import uuid4
from dotenv import load_dotenv
from pydantic import BaseModel, Field

print('All imports OK (packages must be installed via pip beforehand)')

All imports OK (packages must be installed via pip beforehand)


---
## CELL 2 — API key + config

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

# Load .env — searches current dir and parent dirs (optional)
for d in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (d / '.env').exists():
        load_dotenv(d / '.env')
        print(f'Loaded .env from {d}')
        break




print('ANTHROPIC_API_KEY : SET (***hidden***)')
print('No other API keys needed — using ArXiv + DuckDuckGo')
print()

# Config
MODEL           = 'claude-sonnet-4-20250514'
MAX_TOKENS      = 4096
TEMPERATURE     = 0.2
MAX_ITERATIONS  = 18
DOMAIN_CONTEXT  = 'nuclear fuel reprocessing PUREX plutonium France ORANO INB ASN'

# initially 3 domain target 
ACTIVE_DOMAINS = {
    'nuclear_criticality_safety',
    'radiation_protection',
    'process_performance',
}

print(f'Model       : {MODEL}')
print(f'Domains     : {ACTIVE_DOMAINS}')
print(f'Context     : {DOMAIN_CONTEXT}')

Loaded .env from c:\Projects\poc_v1
ANTHROPIC_API_KEY : SET (***hidden***)
No other API keys needed — using ArXiv + DuckDuckGo

Model       : claude-sonnet-4-20250514
Domains     : {'process_performance', 'radiation_protection', 'nuclear_criticality_safety'}
Context     : nuclear fuel reprocessing PUREX plutonium France ORANO INB ASN


---
## CELL 3 — Data models (inline)

In [3]:
class GapSeverity(str, Enum):
    COVERED  = 'covered'
    PARTIAL  = 'partial'
    GAP      = 'gap'
    NO_MATCH = 'no_match'

class TRL(str, Enum):
    TRL1='TRL 1'; TRL2='TRL 2'; TRL3='TRL 3'; TRL4='TRL 4'; TRL5='TRL 5'
    TRL6='TRL 6'; TRL7='TRL 7'; TRL8='TRL 8'; TRL9='TRL 9'; UNKNOWN='unknown'

RESEARCHABLE_CLASSES = {
    'stakeholder_need', 'regulatory_clause', 'engineering_constraint',
    'operational', 'lifecycle', 'failure', 'maintenance', 'degraded',
}

REQ_TYPE_MAP = {
    'stakeholder_need': 'functional', 'regulatory_clause': 'regulatory',
    'engineering_constraint': 'functional', 'operational': 'operational',
    'lifecycle': 'operational', 'failure': 'safety',
    'maintenance': 'maintenance', 'degraded': 'performance',
}

class ResearchItem(BaseModel):
    req_id: str
    entity_class: str
    name: str
    description: Optional[str] = None
    priority: Optional[str] = None
    is_assumption: bool = False
    req_type: str = 'functional'

    def statement(self) -> str:
        return self.description[:300] if self.description else self.name

class StandardMatch(BaseModel):
    name: str
    clause: Optional[str] = None
    verbatim_excerpt: Optional[str] = None
    similarity_score: float = 0.0
    authority_level: Optional[str] = None
    issuing_body: Optional[str] = None
    source_url: Optional[str] = None
    year: Optional[str] = None

class TechnologyMatch(BaseModel):
    name: str
    vendor: Optional[str] = None
    trl: TRL = TRL.UNKNOWN
    description: str = ''
    arxiv_refs: Optional[str] = None
    deployment_examples: Optional[str] = None
    source_url: Optional[str] = None
    limitations: Optional[str] = None

class ResearchRecord(BaseModel):
    req_id: str
    req_statement: str
    requirement_type: str
    source_class: str = 'unknown'
    domain_tag: Optional[str] = None
    criticality: str = 'medium'
    rationale: Optional[str] = None
    standards: List[StandardMatch] = Field(default_factory=list)
    best_standard: Optional[str] = None
    best_standard_clause: Optional[str] = None
    best_similarity_score: float = 0.0
    technologies: List[TechnologyMatch] = Field(default_factory=list)
    top_technology: Optional[str] = None
    top_tech_trl: Optional[str] = None
    gap_severity: GapSeverity = GapSeverity.NO_MATCH
    gap_description: str = ''
    uncovered_aspects: List[str] = Field(default_factory=list)
    recommendation: str = ''
    all_source_urls: List[str] = Field(default_factory=list)
    arxiv_papers_found: int = 0
    web_results_found: int = 0
    researched_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

print('✓ Data models defined')

✓ Data models defined


---
## CELL 4 — Load JSON & extract items for active domains

In [4]:
# Use a stable absolute path for your local machine
json_path = Path(r"c:/Projects/poc_v1/data/engineering_model_iso15926.json")
assert json_path.exists(), f"JSON file not found at {json_path}"

with json_path.open(encoding='utf-8') as f:
    raw = json.load(f)
model_data = raw['iso15926_model']
meta = model_data['meta']

all_ents = (model_data.get('possible_individuals', []) +
            model_data.get('classes', []) +
            model_data.get('things', []))

# Build domain classifier (same logic as the tool) to pre-filter by ACTIVE_DOMAINS
def quick_classify_domain(name, desc):
    t = f'{name} {desc}'.lower()
    if any(w in t for w in ['criticality','fissile','neutron','subcritical']): return 'nuclear_criticality_safety'
    if any(w in t for w in ['radiation','dose','alara','radioprotection','shielding','exposure']): return 'radiation_protection'
    if any(w in t for w in ['yield','throughput','efficiency','extraction','purex','solvent','capacity','processing','reliability']): return 'process_performance'
    if any(w in t for w in ['containment','confinement','leak','release','barrier']): return 'radiological_containment'
    if any(w in t for w in ['control','interface','remote','operator','hmi','scada','dcs']): return 'instrumentation_control'
    if any(w in t for w in ['waste','effluent','discharge','secondary waste']): return 'radioactive_waste_management'
    if any(w in t for w in ['maintenance','repair','telemanipulator']): return 'remote_maintenance'
    if any(w in t for w in ['failure','degraded','fault']): return 'safety_analysis'
    return 'general_nuclear_engineering'

all_items, counter = [], 1
for e in all_ents:
    cls = e.get('class', '')
    if cls not in RESEARCHABLE_CLASSES: continue
    name = e.get('name', '')
    desc = (e.get('description') or e.get('excerpt') or e.get('applicability') or '')
    domain = quick_classify_domain(name, desc)
    all_items.append(ResearchItem(
        req_id=f'REQ-{counter:03d}', entity_class=cls, name=name,
        description=desc[:500] if desc else None,
        priority=e.get('priority'), is_assumption=e.get('is_assumption', False),
        req_type=REQ_TYPE_MAP.get(cls, 'functional'),
    ))
    counter += 1

# Filter to active domains only
active_items = [i for i in all_items
                if quick_classify_domain(i.name, i.description or '') in ACTIVE_DOMAINS]

print(f'Standard   : {meta["standard"]}')
print(f'Total items: {len(all_items)}')
print(f'Active domains: {ACTIVE_DOMAINS}')
print(f'Items in scope : {len(active_items)}')
print()
by_domain = Counter(quick_classify_domain(i.name, i.description or '') for i in active_items)
by_class  = Counter(i.entity_class for i in active_items)
print('By domain:')
for d, n in by_domain.most_common(): print(f'  {d:<35} {n}  {"█"*n}')
print()
print('By class:')
for c, n in by_class.most_common(): print(f'  {c:<25} {n}  {"█"*n}')
print()
print('Items to research:')
for i in active_items:
    dom = quick_classify_domain(i.name, i.description or '')
    print(f'  [{i.req_id}] {i.entity_class:<22} {dom:<32} {i.name[:45]}')

Standard   : ISO/TS 15926-2:2003
Total items: 58
Active domains: {'process_performance', 'radiation_protection', 'nuclear_criticality_safety'}
Items in scope : 31

By domain:
  process_performance                 14  ██████████████
  radiation_protection                10  ██████████
  nuclear_criticality_safety          7  ███████

By class:
  stakeholder_need          10  ██████████
  operational               7  ███████
  lifecycle                 4  ████
  degraded                  3  ███
  maintenance               3  ███
  failure                   3  ███
  regulatory_clause         1  █

Items to research:
  [REQ-001] stakeholder_need       radiation_protection             Worker radioprotection
  [REQ-002] stakeholder_need       process_performance              Process reliability
  [REQ-003] stakeholder_need       process_performance              Annual processing capacity
  [REQ-004] stakeholder_need       process_performance              Radiological containment
  [REQ-005] 

---
## CELL 5 — Search tools (ArXiv API + DuckDuckGo, no keys)

In [5]:
from langchain.tools import tool

SSL_CTX = ssl.create_default_context()


@tool
async def classify_requirement(input_json: str) -> str:
    """
    Classify one requirement into domain_tag, criticality, and targeted
    search queries for standards and technologies.

    Args:
        input_json: JSON string with req_id, req_statement, requirement_type,
                    rationale (optional), domain_context (optional)
    Example:
        {"req_id":"REQ-005","req_statement":"Criticality safety","requirement_type":"functional","rationale":"subcritical fissile","domain_context":"nuclear PUREX"}
    """
    try:
        d = json.loads(input_json)
    except Exception:
        d = {'req_id': 'REQ-?', 'req_statement': input_json,
             'requirement_type': 'functional', 'rationale': '', 'domain_context': ''}

    stmt = d.get('req_statement', '')
    rat  = d.get('rationale', '')
    ctx  = d.get('domain_context', '')
    t    = f'{stmt} {rat} {ctx}'.lower()

    crit = ('high' if any(w in t for w in ['criticality','fissile','radioactive','radiation',
                                            'containment','failure','nuclear safety'])
            else 'medium' if any(w in t for w in ['shall','must','required'])
            else 'low')

    if any(w in t for w in ['criticality','fissile','neutron','subcritical']):
        tag='nuclear_criticality_safety'
        sq='nuclear criticality safety standard fissile material subcritical ANSI ANS-8 IAEA'
        tq='criticality detection neutron detector nuclear reprocessing He-3 BF3 technology'
        aq='nuclear criticality safety detection fissile subcritical monitoring'
    elif any(w in t for w in ['radiation','dose','alara','radioprotection','shielding','exposure']):
        tag='radiation_protection'
        sq='radiation protection occupational exposure nuclear standard IAEA GSR ICRP'
        tq='radiation monitoring dosimetry personal detector nuclear facility technology'
        aq='radiation protection dosimetry nuclear workers occupational exposure'
    elif any(w in t for w in ['yield','throughput','purex','solvent','extraction','reliability','capacity','processing']):
        tag='process_performance'
        sq='PUREX nuclear reprocessing process performance solvent extraction standard IAEA ASTM'
        tq='PUREX pulsed column mixer settler solvent extraction equipment technology nuclear'
        aq='PUREX solvent extraction nuclear reprocessing process optimization performance'
    else:
        tag='general_nuclear_engineering'
        sq=f'nuclear engineering standard {stmt[:80]}'
        tq=f'nuclear technology implementation {stmt[:60]}'
        aq=f'nuclear {stmt[:80]}'

    return json.dumps({
        'req_id': d.get('req_id'), 'domain_tag': tag, 'criticality': crit,
        'search_query_standards': sq, 'search_query_technologies': tq,
        'search_query_arxiv': aq,
        'search_query_fallback': f'nuclear {tag.replace("_"," ")} requirement'
    })


# ── TOOL 2: search_web (DuckDuckGo HTML, no key) ─────────────────────────────
@tool
async def search_web(query: str) -> str:
    """
    Search the web using DuckDuckGo (no API key needed).
    Use for: standards pages, regulatory documents, vendor pages.
    Returns top results with title, URL, and snippet.

    Args:
        query: search query string e.g. "ANSI ANS-8 criticality safety nuclear standard"
    """
    import httpx
    try:
        q = urllib.parse.quote_plus(query)
        url = f'https://html.duckduckgo.com/html/?q={q}'
        async with httpx.AsyncClient(timeout=15.0, follow_redirects=True) as c:
            resp = await c.get(url, headers={
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
                'Accept': 'text/html,application/xhtml+xml',
            })
            html = resp.text

        # Parse results from DDG HTML
        results = []
        # Split on result blocks
        blocks = re.split(r'<div class=["\']result["\']', html)
        for block in blocks[1:8]:  # skip first (header), take up to 7
            title_m   = re.search(r'<a[^>]+class=["\']result__a["\'][^>]*>(.*?)</a>', block, re.DOTALL)
            url_m     = re.search(r'<a[^>]+href=["\']([^"\'>]+)["\']', block)
            snippet_m = re.search(r'class=["\']result__snippet["\'][^>]*>(.*?)</(?:a|span)>', block, re.DOTALL)
            if title_m and url_m:
                title   = re.sub(r'<[^>]+>', '', title_m.group(1)).strip()
                href    = url_m.group(1)
                snippet = re.sub(r'<[^>]+>', ' ', snippet_m.group(1)).strip() if snippet_m else ''
                snippet = re.sub(r'\s+', ' ', snippet)[:300]
                # Clean DDG redirect URLs
                if href.startswith('//duckduckgo.com/l/?'):
                    uddg = re.search(r'uddg=([^&]+)', href)
                    if uddg:
                        href = urllib.parse.unquote(uddg.group(1))
                if href.startswith('http') and title:
                    results.append({'title': title, 'url': href, 'snippet': snippet})
        if not results:
            return json.dumps({'error': 'No results parsed from DDG', 'results': [], 'raw_length': len(html)})
        return json.dumps({'results': results, 'count': len(results)})
    except Exception as exc:
        return json.dumps({'error': str(exc), 'results': []})


# ── TOOL 3: search_arxiv (ArXiv API) ────────────────────────────
@tool
async def search_arxiv(query: str) -> str:
    """
    Search ArXiv for academic papers and technical reports.
    Use for: technology implementations, experimental results, research papers.
    Returns papers with title, authors, abstract, URL, and year.
    No API key needed.

    Args:
        query: search query e.g. "nuclear criticality detection neutron fissile subcritical"
    """
    import httpx
    try:
        q = urllib.parse.quote_plus(query)
        url = f'http://export.arxiv.org/api/query?search_query=all:{q}&max_results=5&sortBy=relevance'
        async with httpx.AsyncClient(timeout=20.0) as c:
            resp = await c.get(url, headers={'User-Agent': 'ResearchAgent/1.0'})
            xml_text = resp.text

        ns = {'a': 'http://www.w3.org/2005/Atom'}
        root = ET.fromstring(xml_text)
        papers = []
        for entry in root.findall('a:entry', ns):
            title   = (entry.findtext('a:title', '', ns) or '').strip().replace('\n', ' ')
            summary = (entry.findtext('a:summary', '', ns) or '').strip().replace('\n', ' ')[:300]
            pid     = entry.findtext('a:id', '', ns)
            pub     = entry.findtext('a:published', '', ns)
            year    = pub[:4] if pub else ''
            authors = [a.findtext('a:name', '', ns)
                       for a in entry.findall('a:author', ns)][:3]
            if title and pid:
                papers.append({'title': title, 'authors': authors,
                               'abstract': summary, 'url': pid, 'year': year})
        if not papers:
            return json.dumps({'papers': [], 'count': 0,
                               'message': 'No ArXiv papers found for this query'})
        return json.dumps({'papers': papers, 'count': len(papers)})
    except Exception as exc:
        return json.dumps({'error': str(exc), 'papers': []})


# ── TOOL 4: fetch_page_content
@tool
async def fetch_page_content(url: str) -> str:
    """
    Fetch the full text of a webpage to get actual clause text or
    detailed technology information. Returns up to 3000 chars of clean text.

    Args:
        url: the full URL to fetch e.g. "https://www.iaea.org/publications/..."
    """
    import httpx
    try:
        async with httpx.AsyncClient(timeout=15.0, follow_redirects=True) as c:
            resp = await c.get(url, headers={'User-Agent': 'Mozilla/5.0'})
            resp.raise_for_status()
            text = re.sub(r'<[^>]+>', ' ', resp.text)
            text = re.sub(r'\s+', ' ', text).strip()
            return text[:3000]
    except Exception as exc:
        return f'Could not fetch {url}: {exc}'


# ── TOOL 5: build_research_record 
@tool
async def build_research_record(record_json: str) -> str:
    """
    Assemble and validate the final research record for one item.
    Call ONCE per item after all searching is done.

    Args:
        record_json: complete JSON with standards[], technologies[],
                     gap_description, recommendation, and all fields.
    """
    try:
        d    = json.loads(record_json)
        stds = d.get('standards', [])
        tech = d.get('technologies', [])
        if stds:
            best = max(stds, key=lambda s: float(s.get('similarity_score', 0)))
            d.setdefault('best_standard',         best.get('name'))
            d.setdefault('best_standard_clause',  best.get('clause'))
            d.setdefault('best_standard_excerpt', best.get('verbatim_excerpt'))
            d.setdefault('best_similarity_score', float(best.get('similarity_score', 0)))
        else:
            d.setdefault('best_similarity_score', 0.0)
        if tech:
            d.setdefault('top_technology',  tech[0].get('name'))
            d.setdefault('top_tech_vendor', tech[0].get('vendor'))
            d.setdefault('top_tech_trl',    tech[0].get('trl'))
        sc = float(d.get('best_similarity_score', 0))
        d['gap_severity'] = ('no_match' if not stds else
                             'covered'  if sc >= 0.80 else
                             'partial'  if sc >= 0.50 else 'gap')
        urls = [s['source_url'] for s in stds if s.get('source_url')]
        urls += [t['source_url'] for t in tech if t.get('source_url')]
        d['all_source_urls'] = list(dict.fromkeys(urls))
        d['researched_at']   = datetime.now(timezone.utc).isoformat()
        return json.dumps({'status': 'ok', 'record': d})
    except Exception as exc:
        return json.dumps({'status': 'error', 'error': str(exc)})


ALL_TOOLS = [classify_requirement, search_web, search_arxiv,
             fetch_page_content, build_research_record]

print('✓ below tools :')
print('  1. classify_requirement  — domain routing + query generation')
print('  2. search_web            — DuckDuckGo HTML (standards, vendors, regs)')
print('  3. search_arxiv          — ArXiv API (papers, tech reports)')
print('  4. fetch_page_content    — full page text from URL')
print('  5. build_research_record — assemble + validate final JSON')

✓ below tools :
  1. classify_requirement  — domain routing + query generation
  2. search_web            — DuckDuckGo HTML (standards, vendors, regs)
  3. search_arxiv          — ArXiv API (papers, tech reports)
  4. fetch_page_content    — full page text from URL
  5. build_research_record — assemble + validate final JSON


---
## CELL 6 — System prompt & agent builder

In [6]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain_anthropic import ChatAnthropic
from langchain.prompts import PromptTemplate

SYSTEM_PROMPT = """You are the Research Agent for a nuclear Systems Engineering platform.

Your task: research ONE engineering item from a nuclear fuel reprocessing system.
Find governing standards AND existing technologies, then produce a precise gap analysis.

MANDATORY STEPS — in order:

STEP 1 — classify_requirement (single JSON string):
  Action Input: {{"req_id": "{req_id}", "req_statement": "{item_name}",
                  "requirement_type": "{req_type}", "rationale": "{item_description}",
                  "domain_context": "{domain_context}"}}

STEP 2 — search_web with search_query_standards (for official standards, regulations):
  Action Input: <search_query_standards from step 1>

STEP 3 — fetch_page_content on the best URL from step 2 (to get real clause text):
  Action Input: <best URL from step 2>

STEP 4 — search_arxiv with search_query_arxiv (for academic/technical papers):
  Action Input: <search_query_arxiv from step 1>

STEP 5 — search_web with search_query_technologies (for vendors, products, implementations):
  Action Input: <search_query_technologies from step 1>

STEP 6 — build_research_record with the COMPLETE assembled JSON:
  Action Input: {{"req_id":"...","req_statement":"...","requirement_type":"...",
    "domain_tag":"...","criticality":"...","rationale":"...","is_assumption":false,
    "standards":[{{"name":"...","clause":"...","verbatim_excerpt":"...",
      "similarity_score":0.0-1.0,"issuing_body":"...","authority_level":"...",
      "source_url":"...","year":"..."}}],
    "technologies":[{{"name":"...","vendor":"...","trl":"TRL N",
      "description":"...","arxiv_refs":"arxiv:XXXX.XXXXX - title",
      "deployment_examples":"...","source_url":"...","limitations":"..."}}],
    "gap_description":"...","uncovered_aspects":["..."],
    "recommendation":"...",
    "search_queries_used":["...","...","..."]}}

SIMILARITY SCORING:
  0.90+ = directly covers this exact item
  0.70-0.90 = same topic with minor scope differences
  0.50-0.70 = related topic, partial coverage
  0.30-0.50 = tangential, significant gaps
  <0.30 = loosely related

TRL: 9=industrial proven, 7-8=pilot, 4-6=lab, 1-3=concept
For technologies, cite ArXiv paper IDs in arxiv_refs field when found.

Domain: nuclear fuel reprocessing / PUREX / France / ORANO / INB / ASN
"""

REACT_TEMPLATE = """{system}

Tools: {tools}\nTool names: {tool_names}

Format EXACTLY:
Thought: <reasoning>
Action: <tool>
Action Input: <input>
Observation: <result>
... (repeat)
Thought: All 6 steps complete.
Final Answer: <JSON string from build_research_record>

Item:
  req_id: {req_id} | source_class: {source_class}
  name: {item_name}
  description: {item_description}
  requirement_type: {req_type} | assumption: {is_assumption}

{agent_scratchpad}"""


def build_agent(verbose=True) -> AgentExecutor:
    llm = ChatAnthropic(
        model=MODEL, api_key=ANTHROPIC_KEY,
        max_tokens=MAX_TOKENS, temperature=TEMPERATURE,
    )
    prompt = PromptTemplate.from_template(REACT_TEMPLATE)
    agent  = create_react_agent(llm=llm, tools=ALL_TOOLS, prompt=prompt)
    return AgentExecutor(
        agent=agent, tools=ALL_TOOLS, verbose=verbose,
        max_iterations=MAX_ITERATIONS, handle_parsing_errors=True,
        return_intermediate_steps=True,
    )

print('✓ System prompt and ReAct template defined')
print('✓ build_agent() ready')

✓ System prompt and ReAct template defined
✓ build_agent() ready


---
## CELL 7 — Output parser

In [7]:
def parse_trl(raw: str) -> TRL:
    norm = str(raw).strip().upper().replace('TRL','TRL ').replace('  ',' ').strip()
    try:    return TRL(norm)
    except: 
        try: return TRL(f'TRL {str(raw).strip()}')
        except: return TRL.UNKNOWN


def parse_output(raw: str, item: ResearchItem, steps: list) -> ResearchRecord:
    """Convert raw agent output → typed ResearchRecord. Never raises."""
    try:
        cleaned = raw.strip()
        if cleaned.startswith('```'):
            cleaned = cleaned.split('```')[1]
            if cleaned.startswith('json'): cleaned = cleaned[4:]
        outer = json.loads(cleaned.strip())
        d = outer.get('record', outer)

        stds = []
        for s in d.get('standards', []):
            try:
                stds.append(StandardMatch(
                    name=s.get('name','?'), clause=s.get('clause'),
                    verbatim_excerpt=s.get('verbatim_excerpt'),
                    similarity_score=float(s.get('similarity_score', 0)),
                    authority_level=s.get('authority_level'),
                    issuing_body=s.get('issuing_body'),
                    source_url=s.get('source_url'), year=s.get('year'),
                ))
            except Exception: pass

        techs = []
        for t in d.get('technologies', []):
            try:
                techs.append(TechnologyMatch(
                    name=t.get('name','?'), vendor=t.get('vendor'),
                    trl=parse_trl(t.get('trl','')),
                    description=t.get('description',''),
                    arxiv_refs=t.get('arxiv_refs'),
                    deployment_examples=t.get('deployment_examples'),
                    source_url=t.get('source_url'),
                    limitations=t.get('limitations'),
                ))
            except Exception: pass

        best  = max(stds, key=lambda s: s.similarity_score) if stds else None
        score = best.similarity_score if best else 0.0
        sev   = (GapSeverity.NO_MATCH if not stds else
                 GapSeverity.COVERED  if score >= 0.80 else
                 GapSeverity.PARTIAL  if score >= 0.50 else GapSeverity.GAP)

        # Count arxiv vs web results from steps
        arxiv_count = sum(1 for a, _ in steps if a.tool == 'search_arxiv')
        web_count   = sum(1 for a, _ in steps if a.tool == 'search_web')

        return ResearchRecord(
            req_id=item.req_id, req_statement=item.name,
            requirement_type=d.get('requirement_type', item.req_type),
            source_class=item.entity_class,
            domain_tag=d.get('domain_tag'),
            criticality=d.get('criticality','medium'),
            rationale=item.description,
            standards=stds,
            best_standard=best.name if best else None,
            best_standard_clause=best.clause if best else None,
            best_similarity_score=round(score, 3),
            technologies=techs,
            top_technology=techs[0].name if techs else None,
            top_tech_trl=techs[0].trl.value if techs else None,
            gap_severity=sev,
            gap_description=d.get('gap_description',''),
            uncovered_aspects=d.get('uncovered_aspects',[]),
            recommendation=d.get('recommendation',''),
            all_source_urls=d.get('all_source_urls',[]),
            arxiv_papers_found=arxiv_count,
            web_results_found=web_count,
        )
    except Exception as exc:
        print(f'  ⚠ parse failed ({item.req_id}): {exc}')
        return ResearchRecord(
            req_id=item.req_id, req_statement=item.name,
            requirement_type=item.req_type, source_class=item.entity_class,
            rationale=item.description, gap_severity=GapSeverity.NO_MATCH,
            gap_description=f'Parse error: {exc}',
            recommendation='Manual research required.',
        )

print('✓ parse_output() defined')

✓ parse_output() defined


---
## CELL 8 — DEMO: Run agent on ONE item (full verbose trace)

Run this first. You will see every `Thought / Action / Observation` step printed live.

In [8]:
# Pick the demo item — REQ-005 Criticality safety is the richest for this domain
demo_item = next(i for i in active_items if 'riticality' in i.name)

print('=' * 72)
print(f'DEMO — {demo_item.req_id}  [{demo_item.entity_class}]')
print(f'Name : {demo_item.name}')
print(f'Desc : {(demo_item.description or "")[:180]}')
print('=' * 72)
print('Starting ReAct agent... (verbose=True)\n')

agent = build_agent(verbose=True)

demo_result = await agent.ainvoke({
    'system':           SYSTEM_PROMPT.format(
                            req_id=demo_item.req_id, item_name=demo_item.name,
                            item_description=(demo_item.description or '')[:300],
                            req_type=demo_item.req_type, domain_context=DOMAIN_CONTEXT,
                        ),
    'req_id':           demo_item.req_id,
    'source_class':     demo_item.entity_class,
    'item_name':        demo_item.name,
    'item_description': (demo_item.description or '')[:300],
    'req_type':         demo_item.req_type,
    'is_assumption':    str(demo_item.is_assumption).lower(),
    'agent_scratchpad': '',
})

demo_record = parse_output(
    demo_result['output'],
    demo_item,
    demo_result.get('intermediate_steps', [])
)

# ── Print tool call trace ──────────────────────────────────────────────────────
steps = demo_result.get('intermediate_steps', [])
print()
print('=' * 72)
print(f'TOOL CALL TRACE ({len(steps)} steps)')
print('=' * 72)
for i, (action, obs) in enumerate(steps, 1):
    print(f'\nStep {i}: {action.tool}')
    inp = str(action.tool_input)
    print(f'  Input  : {inp[:120]}')
    ob  = str(obs)
    try:
        ob_data = json.loads(ob)
        if 'results' in ob_data:
            print(f'  Output : {len(ob_data["results"])} web results')
            for r in ob_data['results'][:2]:
                print(f'    → {r.get("title","")[:60]}')
                print(f'      {r.get("url","")[:80]}')
        elif 'papers' in ob_data:
            print(f'  Output : {len(ob_data["papers"])} ArXiv papers')
            for p in ob_data['papers'][:2]:
                print(f'    → {p.get("title","")[:60]} ({p.get("year","")})')
                print(f'      {p.get("url","")[:70]}')
        elif 'domain_tag' in ob_data:
            print(f'  Output : domain={ob_data["domain_tag"]} | crit={ob_data["criticality"]}')
        elif ob_data.get('status') == 'ok':
            print(f'  Output : record built OK')
        else:
            print(f'  Output : {ob[:120]}')
    except:
        print(f'  Output : {ob[:120]}')

# ── Print parsed result ────────────────────────────────────────────────────────
SEV = {'covered':'✓ COVERED','partial':'~ PARTIAL','gap':'✗ GAP','no_match':'? NO MATCH'}
print()
print('=' * 72)
print('PARSED RESULT')
print('=' * 72)
print(f'  {demo_record.req_id} | {demo_record.source_class} | {demo_record.domain_tag}')
print(f'  Criticality : {demo_record.criticality}')
print(f'  Gap status  : {SEV[demo_record.gap_severity.value]}')
print(f'  Best score  : {demo_record.best_similarity_score}')
print(f'  Web searches: {demo_record.web_results_found} | ArXiv searches: {demo_record.arxiv_papers_found}')
print()
print(f'  STANDARDS ({len(demo_record.standards)}):')
for s in demo_record.standards:
    bar = '█' * int(s.similarity_score * 20)
    print(f'    [{s.similarity_score:.2f}] {bar:<20} {s.name}')
    if s.clause:           print(f'           clause  : {s.clause}')
    if s.verbatim_excerpt: print(f'           excerpt : {s.verbatim_excerpt[:130]}')
    if s.source_url:       print(f'           url     : {s.source_url}')
print()
print(f'  TECHNOLOGIES ({len(demo_record.technologies)}):')
for t in demo_record.technologies:
    print(f'    [{t.trl.value}] {t.name} — {t.vendor}')
    print(f'           {t.description[:130]}')
    if t.arxiv_refs:           print(f'           arxiv: {t.arxiv_refs[:100]}')
    if t.deployment_examples:  print(f'           deploy: {t.deployment_examples[:100]}')
print()
print(f'  GAP  : {demo_record.gap_description[:300]}')
print()
if demo_record.uncovered_aspects:
    print('  UNCOVERED:')
    for a in demo_record.uncovered_aspects: print(f'    ✗ {a}')
print()
print(f'  RECOMMENDATION:')
print(f'    {demo_record.recommendation[:350]}')

DEMO — REQ-005  [stakeholder_need]
Name : Criticality safety
Desc : The subsystem shall ensure that the extraction and handling of fissile materials remain subcritical under all operational and credible accident conditions.
Starting ReAct agent... (verbose=True)



> Entering new AgentExecutor chain...
Thought: I need to research criticality safety for nuclear fuel reprocessing systems. Let me start by classifying this requirement to understand the domain context and generate targeted search queries.

Action: classify_requirement
Action Input: {"req_id": "REQ-005", "req_statement": "Criticality safety", "requirement_type": "functional", "rationale": "The subsystem shall ensure that the extraction and handling of fissile materials remain subcritical under all operational and credible accident conditions.", "domain_context": "nuclear fuel reprocessing PUREX plutonium France ORANO INB ASN"}
{"req_id": "REQ-005", "domain_tag": "nuclear_criticality_safety", "criticality": "high", "search_qu

---
## CELL 9 — Run all items in active domains

⏱ **~2–3 min per item** (LLM + web searches). Progress printed after each.

In [10]:
all_records: List[ResearchRecord] = [demo_record]   # start with demo result
done_ids = {demo_record.req_id}
remaining = [i for i in active_items if i.req_id not in done_ids]

print(f'Already done : {list(done_ids)}')
print(f'Remaining    : {len(remaining)} items')
print()

agent = build_agent(verbose=False)   # silent for bulk run
t0 = time.time()

for idx, item in enumerate(remaining, 1):
    dom = quick_classify_domain(item.name, item.description or '')
    t1  = time.time()
    print(f'[{idx:02d}/{len(remaining)}] {item.req_id} | {item.entity_class:<22} | {item.name[:50]}')

    try:
        res = await agent.ainvoke({
            'system':           SYSTEM_PROMPT.format(
                                    req_id=item.req_id, item_name=item.name,
                                    item_description=(item.description or '')[:300],
                                    req_type=item.req_type, domain_context=DOMAIN_CONTEXT,
                                ),
            'req_id':           item.req_id,
            'source_class':     item.entity_class,
            'item_name':        item.name,
            'item_description': (item.description or '')[:300],
            'req_type':         item.req_type,
            'is_assumption':    str(item.is_assumption).lower(),
            'agent_scratchpad': '',
        })
        steps  = res.get('intermediate_steps', [])
        record = parse_output(res['output'], item, steps)
        n_steps = len(steps)
    except Exception as exc:
        print(f'  ✗ Error: {exc}')
        record  = ResearchRecord(
            req_id=item.req_id, req_statement=item.name,
            requirement_type=item.req_type, source_class=item.entity_class,
            rationale=item.description, gap_severity=GapSeverity.NO_MATCH,
            gap_description=f'Agent error: {exc}',
            recommendation='Manual research required.',
        )
        n_steps = 0

    all_records.append(record)
    elapsed = time.time() - t1

    SEV_ICON = {'covered':'✓','partial':'~','gap':'✗','no_match':'?'}
    icon = SEV_ICON[record.gap_severity.value]
    print(f'  {icon} {record.gap_severity.value:<8} score={record.best_similarity_score:.2f}  '
          f'steps={n_steps}  web={record.web_results_found}  arxiv={record.arxiv_papers_found}  '
          f'({elapsed:.0f}s)')
    print(f'    std : {(record.best_standard or "none")[:60]}')
    print(f'    tech: {(record.top_technology or "none")[:50]} ({record.top_tech_trl or "-"})')
    print()

print(f'Done — {len(all_records)} items in {(time.time()-t0)/60:.1f} min')

Already done : ['REQ-005']
Remaining    : 30 items

[01/30] REQ-001 | stakeholder_need       | Worker radioprotection
  ✓ covered  score=0.95  steps=8  web=5  arxiv=1  (59s)
    std : IAEA GSR Part 3
    tech: Remote Handling Systems (TRL 9)

[02/30] REQ-002 | stakeholder_need       | Process reliability
  ✓ covered  score=0.90  steps=8  web=5  arxiv=1  (59s)
    std : IAEA NS-G-1.12
    tech: PUREX Pulsed Columns (TRL 9)

[03/30] REQ-003 | stakeholder_need       | Annual processing capacity
  ~ partial  score=0.75  steps=7  web=4  arxiv=1  (51s)
    std : IAEA Safety Standards Series No. SSG-42
    tech: PUREX Pulsed Column System (TRL 9)

[04/30] REQ-004 | stakeholder_need       | Radiological containment
  ✓ covered  score=0.95  steps=9  web=6  arxiv=1  (71s)
    std : IAEA SSR-4
    tech: Multi-barrier containment system (TRL 9)

[05/30] REQ-007 | stakeholder_need       | Process scalability
  ~ partial  score=0.75  steps=8  web=5  arxiv=1  (52s)
    std : IAEA SSR-4
    tech: PURE


## CELL 10 — Summary table + stats

In [ ]:
SEV_ICON = {'covered':'✓','partial':'~','gap':'✗','no_match':'?'}
SEV_LABEL = {'covered':'COVERED','partial':'PARTIAL','gap':'GAP','no_match':'NO MATCH'}

print('=' * 110)
print('FULL SUMMARY TABLE')
print('=' * 110)
print(f'{"REQ":<8} {"CLASS":<22} {"SCORE":>6}  {"GAP":<10} {"BEST STANDARD":<38} {"TOP TECH":<28} {"TRL":<8}')
print('-' * 110)
for r in sorted(all_records, key=lambda x: -x.best_similarity_score):
    icon  = SEV_ICON[r.gap_severity.value]
    sev   = SEV_LABEL[r.gap_severity.value]
    std   = (r.best_standard or 'none')[:37]
    tech  = (r.top_technology or 'none')[:27]
    trl   = (r.top_tech_trl or '-')[:7]
    print(f'{r.req_id:<8} {r.source_class:<22} {r.best_similarity_score:>6.2f}  '
          f'{icon} {sev:<8} {std:<38} {tech:<28} {trl}')

# ── Stats ──────────────────────────────────────────────────────────────────────
n       = len(all_records)
covered = sum(1 for r in all_records if r.gap_severity == GapSeverity.COVERED)
partial = sum(1 for r in all_records if r.gap_severity == GapSeverity.PARTIAL)
gap     = sum(1 for r in all_records if r.gap_severity == GapSeverity.GAP)
nm      = sum(1 for r in all_records if r.gap_severity == GapSeverity.NO_MATCH)
scored  = [r for r in all_records if r.best_similarity_score > 0]
avg     = round(sum(r.best_similarity_score for r in scored) / max(len(scored),1), 3)

all_stds  = [s for r in all_records for s in r.standards]
all_techs = [t for r in all_records for t in r.technologies]

print()
print('STATS')
print(f'  Total items   : {n}')
print(f'  ✓ Covered     : {covered}   (score ≥ 0.80)')
print(f'  ~ Partial     : {partial}   (0.50 – 0.79)')
print(f'  ✗ Gap         : {gap}   (score < 0.50)')
print(f'  ? No match    : {nm}')
print(f'  Avg score     : {avg}')
print(f'  Standards     : {len(set(s.name for s in all_stds))} unique')
print(f'  Technologies  : {len(set(t.name for t in all_techs))} unique')
print(f'  ArXiv papers  : {sum(r.arxiv_papers_found for r in all_records)} searches done')
print()

print('Standards by issuing body:')
bodies = Counter(s.issuing_body or 'OTHER' for s in all_stds)
for b, c in bodies.most_common():
    print(f'  {b:<12} {c:>3}  {"█"*c}')
print()

print('Technology TRL breakdown:')
trls = Counter(t.trl.value for t in all_techs)
for trl, c in sorted(trls.items()):
    print(f'  {trl:<10} {c:>3}  {"█"*c}')
print()

print('Critical gaps requiring action:')
critical = [r for r in all_records
            if r.gap_severity in (GapSeverity.GAP, GapSeverity.NO_MATCH)]
for r in critical:
    print(f'  ✗ {r.req_id} [{r.source_class}]: {r.gap_description[:80]}')


## CELL 11 — Inspect one full record in detail

In [ ]:
# Change to inspect any record by req_id
INSPECT = 'REQ-005'   # change e.g. to 'REQ-001'

rec = next((r for r in all_records if r.req_id == INSPECT), all_records[0])

W = 72
print('═' * W)
print(f'  {rec.req_id}  [{rec.source_class}]')
print('═' * W)
print(f'  Name          : {rec.req_statement}')
print(f'  Domain        : {rec.domain_tag}')
print(f'  Criticality   : {rec.criticality}')
print(f'  Gap status    : {rec.gap_severity.value.upper()}')
print(f'  Best score    : {rec.best_similarity_score}')
if rec.rationale:
    print()
    for line in textwrap.wrap(rec.rationale[:350], 68):
        print(f'  {line}')

print()
print(f'  STANDARDS FOUND ({len(rec.standards)}) — from DuckDuckGo web search')
print('  ' + '─' * (W-2))
for i, s in enumerate(rec.standards, 1):
    bar = '█' * int(s.similarity_score * 20)
    print(f'  [{i}] {s.name}')
    print(f'      Body      : {s.issuing_body}  |  Level: {s.authority_level}')
    print(f'      Year      : {s.year}')
    print(f'      Clause    : {s.clause}')
    print(f'      Score     : {s.similarity_score:.2f}  {bar}')
    if s.verbatim_excerpt:
        print(f'      Excerpt   : {s.verbatim_excerpt[:160]}')
    if s.source_url:
        print(f'      URL       : {s.source_url}')
    print()

print(f'  TECHNOLOGIES FOUND ({len(rec.technologies)}) — from DuckDuckGo + ArXiv')
print('  ' + '─' * (W-2))
for i, t in enumerate(rec.technologies, 1):
    print(f'  [{i}] {t.name}')
    print(f'      Vendor    : {t.vendor}')
    print(f'      TRL       : {t.trl.value}')
    for line in textwrap.wrap(t.description[:250], 64):
        print(f'      Desc      : {line}')
    if t.arxiv_refs:
        print(f'      ArXiv     : {t.arxiv_refs[:120]}')
    if t.deployment_examples:
        print(f'      Deployed  : {t.deployment_examples[:120]}')
    if t.limitations:
        print(f'      Limits    : {t.limitations[:120]}')
    if t.source_url:
        print(f'      URL       : {t.source_url}')
    print()

print(f'  GAP ANALYSIS')
print('  ' + '─' * (W-2))
for line in textwrap.wrap(rec.gap_description, 68):
    print(f'  {line}')
print()
if rec.uncovered_aspects:
    print('  Uncovered aspects:')
    for a in rec.uncovered_aspects: print(f'    ✗ {a}')
    print()
print('  RECOMMENDATION')
print('  ' + '─' * (W-2))
for line in textwrap.wrap(rec.recommendation, 68):
    print(f'  {line}')
if rec.all_source_urls:
    print()
    print('  All source URLs:')
    for u in rec.all_source_urls: print(f'    • {u}')


## CELL 12 — Save results to JSON + CSV

In [ ]:
import csv

out_dir = Path.cwd()
if (out_dir/'data').exists(): out_dir = out_dir/'data'
elif (out_dir.parent/'data').exists(): out_dir = out_dir.parent/'data'

ts = datetime.now().strftime('%Y%m%d_%H%M')

# ── Full JSON ─────────────────────────────────────────────────────────────────
full_out = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'source_standard': meta['standard'],
    'domains_researched': list(ACTIVE_DOMAINS),
    'total_items': len(all_records),
    'stats': {'covered': covered, 'partial': partial, 'gaps': gap,
              'no_match': nm, 'avg_similarity': avg},
    'records': [r.model_dump() for r in all_records]
}
fp_json = out_dir / f'research_results_{ts}.json'
with fp_json.open('w', encoding='utf-8') as f:
    json.dump(full_out, f, indent=2, ensure_ascii=False)
print(f'✓ JSON saved : {fp_json}  ({fp_json.stat().st_size//1024} KB)')

# ── CSV (flat summary) ────────────────────────────────────────────────────────
fp_csv = out_dir / f'research_summary_{ts}.csv'
rows = [
    {
        'req_id':           r.req_id,
        'source_class':     r.source_class,
        'domain_tag':       r.domain_tag or '',
        'criticality':      r.criticality,
        'req_statement':    r.req_statement[:100],
        'best_standard':    r.best_standard or '',
        'standard_clause':  r.best_standard_clause or '',
        'similarity_score': r.best_similarity_score,
        'gap_severity':     r.gap_severity.value,
        'technologies_count': len(r.technologies),
        'top_technology':   r.top_technology or '',
        'top_trl':          r.top_tech_trl or '',
        'arxiv_papers':     r.arxiv_papers_found,
        'gap_description':  r.gap_description[:200],
        'recommendation':   r.recommendation[:200],
        'all_standards':    '; '.join(s.name for s in r.standards),
        'source_urls':      '; '.join(r.all_source_urls),
    }
    for r in all_records
]
with fp_csv.open('w', newline='', encoding='utf-8-sig') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print(f'✓ CSV saved  : {fp_csv}')
print()
print('All done!')